# RECAP FREE — Kaggle

ဒီ notebook က GitHub `main` branch မှ project ကို clone လုပ်ပြီး run ပါမယ်။ API key ကို code/Secrets ထဲ မထည့်ပါနှင့်။ Server UI ပွင့်လာပြီးမှ **Settings** ထဲမှာ ယာယီထည့်ပါ။

**Kaggle options:** Internet ON; VoxCPM2 သုံးမယ်ဆို GPU ON။


In [ ]:
# CELL 1 — GitHub main branch ကို clone လုပ်မယ်
from pathlib import Path
import shutil, subprocess

project = Path('/kaggle/working/recap-free')
shutil.rmtree(project, ignore_errors=True)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/surviveman78-commits/recap-free.git', str(project)], check=True)
print('Project cloned:', project)
print('Latest source:', subprocess.check_output(['git', '-C', str(project), 'log', '-1', '--oneline'], text=True).strip())


In [ ]:
# CELL 2 — FFmpeg, Myanmar fonts, and all dependencies
%cd /kaggle/working/recap-free
!apt-get update -qq
!apt-get install -y -qq ffmpeg fontconfig fonts-noto-core fonts-sil-padauk fonts-myanmar libsndfile1
!fc-cache -f
!pip install -q -r requirements-kaggle.txt
print('Cloud + Local AI dependencies ready')


In [ ]:
# CELL 3 — Local AI models ကို UI မဖွင့်ခင် တစ်ခါတည်း download လုပ်မယ်
%cd /kaggle/working/recap-free
from huggingface_hub import snapshot_download
import os
models_root = '/kaggle/working/models'
voxcpm_dir = f'{models_root}/VoxCPM2'
whisper_dir = f'{models_root}/faster-whisper-large-v3'
nllb_dir = f'{models_root}/nllb-200-distilled-1.3B'
snapshot_download(repo_id='openbmb/VoxCPM2', local_dir=voxcpm_dir)
snapshot_download(repo_id='Systran/faster-whisper-large-v3', local_dir=whisper_dir)
snapshot_download(repo_id='facebook/nllb-200-distilled-1.3B', local_dir=nllb_dir)
os.environ['VOXCPM_MODEL_ID'] = voxcpm_dir
os.environ['RECAP_WHISPER_MODEL'] = whisper_dir
os.environ['RECAP_NLLB_MODEL'] = nllb_dir
os.environ['RECAP_LOCAL_ONLY'] = '1'
os.environ['RECAP_WHISPER_DEVICE'] = 'cuda'
os.environ['RECAP_WHISPER_DEVICE_INDEX'] = '0'
os.environ['RECAP_NLLB_DEVICE'] = 'cuda:1'
print('All local models downloaded and paths exported.')


In [ ]:
# CELL 4 — server + public URL
# Local AI mode ကို default ထားထားပြီး API key မလိုပါ။
%cd /kaggle/working/recap-free
!python kaggle_foreground.py


In [ ]:
# CELL 5 — local health check
import urllib.request
with urllib.request.urlopen('http://127.0.0.1:8000/api/settings', timeout=15) as r:
    print('Studio HTTP status:', r.status)
print('UI Settings > AI Mode = Local AI ထားပါ။ Cloud API မလိုပါ။')


## အသုံးပြုရန်

1. Cell 1 → Cell 2 → Cell 4 → Cell 5 ကို အစဉ်လိုက် run ပါ။
2. VoxCPM2 သုံးမယ်ဆို Cell 3 ကို Cell 4 မတိုင်ခင် run ပါ။
3. UI ပွင့်လာရင် **Settings** ထဲမှာ API key နှစ်ခုထည့်ပြီး Save လုပ်ပါ။
4. မြန်မာစာတန်းထိုးအတွက် Font ကို **Noto Sans Myanmar** သို့မဟုတ် **Padauk** ရွေးပါ။
5. Edge TTS က Internet လိုပြီး VoxCPM2 က GPU/Internet ပထမ download လိုပါတယ်။
